In [ ]:
import sys; print(sys.executable)

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from data.load import load_split
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from transformers import AutoTokenizer


In [ ]:
train, val, test = load_split("standard")

In [ ]:
df = pd.concat(
    [
        train.assign(split="train"),
        val.assign(split="val"),
        test.assign(split="test")
    ],
    ignore_index=True,
)

tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")


N_SPECIAL = tok.num_special_tokens_to_add(pair=True) + 1

print(f"{len(df):,} rows | max length {tok.model_max_length:,} | {N_SPECIAL} special tokens")

In [ ]:
FIELDS = ["criterion", "completion", "conversation"]

def n_tokens(col: pd.Series) -> pd.Series:

    unq = col.unique().tolist()
    ids = tok(unq, add_special_tokens=False)["input_ids"]

    return col.map(dict(zip(unq, map(len, ids))))


for f in FIELDS:
    df[f"{f}_n_tokens"] = n_tokens(df[f])

df["len_total"] = df[[f"{f}_n_tokens" for f in FIELDS]].sum(axis=1) + N_SPECIAL

LEN_COLS = [f"{f}_n_tokens" for f in FIELDS] + ["len_total"]

df.loc[df.split == "train", LEN_COLS].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0)


    

In [ ]:
crit = (
    df.groupby("criterion")
    .agg(
        tokens=("criterion_n_tokens", "first"),
        rows=("criterion", "size"),
        pct_met=("hard_label", "mean"),
    )
    .sort_values(by="tokens", ascending=False)
)

crit["pct_met"] = (100 * crit.pct_met).round(1)
crit.index = crit.index.str.slice(0, 80)
crit


In [ ]:
BUDGETS = [512, 1024, 2048, 4096, 8192]

def coverage(frame: pd.DataFrame) -> pd.DataFrame:

    fixed = frame.criterion_n_tokens + frame.completion_n_tokens + N_SPECIAL

    out = []

    for b in BUDGETS:

        room = (b - fixed).clip(lower=0)
        lost = (frame.conversation_n_tokens - room).clip(lower=0)

        cut = lost>0
        out.append({
            "budget": b,
            "first_%": 100 * (frame.len_total <= b).mean(),
            "conv_cut_%": 100 * cut.mean(),
            "no_room_%": 100 * (fixed > b).mean(),
            "median_tokens_lost_when_cut": lost[cut].median() if cut.any() else 0,
            "conv_tokens_kept_%": 100 * (1 - lost.sum() / frame.conversation_n_tokens.sum()),

        })

    return pd.DataFrame(out).set_index("budget").round(2)

coverage(df[df.split == "train"])

    

In [ ]:
LEN_COLS = [f"{f}_n_tokens" for f in FIELDS] + ["len_total"]
tr = df[df.split == "train"]

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))

for ax, col in zip(axes, LEN_COLS):
    ax.hist(tr[col], bins=60)
    ax.set_title(col)
    ax.set_xlabel("tokens")

for b in(1024, 2048):
    axes[-1].axvline(b, color="k", ls="--", lw=1)
    axes[-1].text(b, axes[-1].get_ylim()[1] * 0.9, f" {b}", fontsize=9)


axes[0].set_ylabel("rows")
plt.tight_layout()

In [ ]:
rows = []

for b in (1024, 2048):

    for over, g in tr.groupby(tr.len_total > b):

        rows.append({

            "budget": b,
            "group": f"> {b}" if over else f"<= {b}",
            "rows": len(g),
            "pct_met": 100 * g.hard_label.mean(),
            "pct_disagree": 100 * (~g.unanimous).mean(),
            
        })

pd.DataFrame(rows).set_index(["budget", "group"]).round(1)


In [ ]:
df.groupby("split").agg(
    rows=("len_total", "size"),
    p50=("len_total", "median"),
    p95=("len_total", lambda s: s.quantile(0.95)),
    max=("len_total", "max"),
    fits_2048_pct=("len_total", lambda s: 100 * (s <= 2048).mean()),
).loc[["train", "val", "test"]].round(1)


## Summary

1. **Objective:** choose `max_len` and a truncation rule for the encoder judge. The input is `[CLS] criterion [SEP] completion [SEP] conversation [SEP]`. Lengths are measured with the ModernBERT-base tokenizer on all 29,511 rows.
2. **Observations:** total input is a median of 829 tokens, a 95th percentile of 1,752 and a maximum of 4,088. 98.1% of rows fit in 2,048 tokens, but only 25% fit in 512. Criteria range from 9 to 601 tokens, and their met rates range from 53% to 97.5%. Val and test match train within about a point.
3. **Inference:** standard 512-token encoders (BERT, PubMedBERT, ClinicalBERT) would cut most inputs, so a long-context encoder is required. Rows over 2,048 tokens are 93.5% met, so truncation mostly affects easy examples. The criterion alone is predictive, so the encoder has to beat a per-criterion base-rate baseline, not just always-met.
4. **Decision:** `max_len = 2048`. Never cut the criterion. Cut the conversation from the left, keeping the latest turns. In the 0.85% of rows where criterion plus completion alone exceed 2,048, drop the conversation and cut the end of the completion. `max_len = 4096` is an optional ablation with no truncation at all.
